### Hyper-parameter fine tuning of top 3 models

1. LightGBM
2. Catboost
3. XGBoost

In [4]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np

import mlflow
import optuna
import xgboost as xgb
import lightgbm as lgb

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from feature_engine.selection import DropConstantFeatures
from src.data.features import MutualInfoSelector

from src.data.loading import loading, sample 
from src.data.cleaning import cleaning 
from src.data.splitting import cross_val_split


/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
raw_df = loading(file_path='/Users/thananpornsethjinda/Desktop/credit-risk-modeling/data/accepted_2007_to_2018Q4.csv')
raw_sample = sample(df=raw_df)
cleaned_sample = cleaning(df=raw_sample)

Reading and loading file ...


/Users/thananpornsethjinda/Desktop/credit-risk-modeling/src/data/loading.py:12: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv(file_path)


Data successfully read in 39.009411096572876 seconds!
Starting data cleaning
Grouping target variable to binary targets (Charged Off) and (Fully Paid) ...
Dropping loan status null values
A total of 45 were dropped; with the columns being ['member_id', 'desc', 'mths_since_last_record', 'next_pymnt_d', 'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'mths_since_rcnt_il', 'il_util', 'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date'

In [7]:
X_train_val, X_test, y_train_val, y_test = cross_val_split(df=cleaned_sample)

from sklearn import set_config

# This forces all transformers to output DataFrames instead of Numpy arrays
set_config(transform_output="pandas")

NUMERICAL_FEATURES = X_train_val.select_dtypes(include=['float64']).columns

ONE_HOT_FEATURES = ['term', 'home_ownership']

ORDINAL_FEATURES = ['sub_grade']

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaling", RobustScaler())
    ]
)

CATEGORICAL_FEATURES = [
    col for col in X_train_val.select_dtypes(exclude=['float64']).columns 
    if col not in ONE_HOT_FEATURES and col not in ORDINAL_FEATURES
]

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ]
)

one_hot_transformer = Pipeline(
    steps=[
        ("one-hot-encoding", OneHotEncoder(drop="first", sparse_output=False))
    ]
)


ordinal_transformer = Pipeline(
    steps=[
        ("ordinal-encoding", OrdinalEncoder())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, NUMERICAL_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
        ("cat_one_hot", one_hot_transformer, ONE_HOT_FEATURES),
        ("cat_ordinal", ordinal_transformer, ORDINAL_FEATURES)
    ]
)

In [8]:
from sklearn.metrics import fbeta_score
from sklearn.model_selection import cross_validate, StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, precision_score, recall_score, make_scorer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def train(pipeline): 

    f1 = make_scorer(f1_score, average='weighted')
    precision = make_scorer(precision_score, pos_label=1)
    recall = make_scorer(recall_score, pos_label=1)
    f2 = make_scorer(fbeta_score, average='weighted', beta=2)
    scoring = {'f1': f1, 'precision':precision, 'recall':recall, 'f2': f2}

    y_train_numeric = y_train_val.map({'Fully Paid': 0, 'Charged Off': 1})
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(pipeline, X_train_val, y_train_numeric, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)
    print(pd.DataFrame(scores).mean())
    avg_scores = pd.DataFrame(scores).mean().to_dict()
    mlflow.log_metrics(avg_scores)

    y_pred = cross_val_predict(pipeline, X_train_val, y_train_numeric, cv=cv, n_jobs=-1)
    cm = confusion_matrix(y_train_numeric, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    mlflow.log_figure(disp.figure_, "confusion_matrix.png")

    print("Done!")


In [11]:
y_train_numeric = y_train_val.map({'Fully Paid': 0, 'Charged Off': 1})


In [9]:
def plot_feature_importance(pipeline, model_type):
  """
  Plots feature importance for an XGBoost model.

  Args:
  - model: A trained XGBoost model

  Returns:
  - fig: The matplotlib figure object
  """
  fig, ax = plt.subplots(figsize=(10, 8))
  importance_type = "gain"

  model_type.plot_importance(
      pipeline[-1],
      importance_type=importance_type,
      ax=ax,
      title=f"Feature Importance based on {importance_type}",
  )
  plt.tight_layout()
  plt.close(fig)

  return fig

In [12]:
# override Optuna's default logging to ERROR only
optuna.logging.set_verbosity(optuna.logging.ERROR)

# define a logging callback that will report on only new challenger parameter configurations if a
# trial has usurped the state of 'best conditions'


def champion_callback(study, frozen_trial):
  """
  Logging callback that will report when a new trial iteration improves upon existing
  best trial values.

  Note: This callback is not intended for use in distributed computing systems such as Spark
  or Ray due to the micro-batch iterative implementation for distributing trials to a cluster's
  workers or agents.
  The race conditions with file system state management for distributed trials will render
  inconsistent values with this callback.
  """

  winner = study.user_attrs.get("winner", None)

  if study.best_value and winner != study.best_value:
      study.set_user_attr("winner", study.best_value)
      if winner:
          improvement_percent = (abs(winner - study.best_value) / study.best_value) * 100
          print(
              f"Trial {frozen_trial.number} achieved value: {frozen_trial.value} with "
              f"{improvement_percent: .4f}% improvement"
          )
      else:
          print(f"Initial trial {frozen_trial.number} achieved value: {frozen_trial.value}")

## LightGBM

In [13]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("lightgbm-tuning")

2026/03/08 11:50:25 INFO mlflow.tracking.fluent: Experiment with name 'lightgbm-tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/thananpornsethjinda/Desktop/credit-risk-modeling/notebooks/mlruns/3', creation_time=1772945425106, experiment_id='3', last_update_time=1772945425106, lifecycle_stage='active', name='lightgbm-tuning', tags={}, workspace='default'>

In [14]:
def objective(trial): 
    with mlflow.start_run(nested=True): 
        # define lightgbm hyperparameters 
        params = {
            "verbosity": -1, 
            "random_state": 42, 
            "n_jobs": -1,
            "class_weight": "balanced", 
            "num_leaves": trial.suggest_int("num_leaves", 20, 40),
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "min_child_samples": trial.suggest_int("min_child_samples", 15, 20)
        }

        model = LGBMClassifier(**params)

        pipeline = Pipeline(
                    steps=[
                        ("preprocessing", preprocessor), 
                        ("quasi-constant feature dropping", DropConstantFeatures(tol=0.95)),  
                        ("mutual-information selector", MutualInfoSelector(k=10, categorical_indices=[45, 46, 47, 48, 49])), 
                        ("model", model)
                    ]
                )
        
        y_train_numeric = y_train_val.map({'Fully Paid': 0, 'Charged Off': 1})
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_validate(pipeline, X_train_val, y_train_numeric, cv=cv, scoring='recall', return_train_score=True, n_jobs=-1)
        test_recall = scores['test_score'].mean() 

        avg_scores = pd.DataFrame(scores).mean().to_dict()

        mlflow.log_metrics(avg_scores)
        mlflow.log_params(params)
        mlflow.log_param("model type", "LightGBM Classifier")

        return test_recall # return recall value 

In [15]:
run_name="LightGBM"
with mlflow.start_run(run_name=run_name): 

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=3, callbacks=[champion_callback])
    print("Optimisation works!")

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_test_recall", study.best_value)

    mlflow.set_tags(
        tags={
            "project_name": "credit risk modeling",
            "model_family": "lightgbm",
            "optimizer_engine": "optuna"
        }
    )

    model = LGBMClassifier(**study.best_params)

    final_pipeline = Pipeline(
                    steps=[
                        ("preprocessing", preprocessor), 
                        ("quasi-constant feature dropping", DropConstantFeatures(tol=0.95)),  
                        ("mutual-information selector", MutualInfoSelector(k=10, categorical_indices=[45, 46, 47, 48, 49])), 
                        ("model", model)
                    ]
                )
    
    final_pipeline.fit(X_train_val, y_train_numeric)

    importances = plot_feature_importance(final_pipeline, model_type=lgb)
    mlflow.log_figure(figure=importances, artifact_file="feature_importances.png")

    artifact_path = "model"

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="model_pipeline",
        input_example=X_train_val.iloc[:5] # Useful for UI preview
    )

    model_uri = mlflow.get_artifact_uri(artifact_path)


Initial trial 0 achieved value: 0.6409166628092876
Trial 1 achieved value: 0.6557100627743931 with  2.2561% improvement
Optimisation works!
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 26498, number of negative: 96907
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002100 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1503
[LightGBM] [Info] Number of data points in the train set: 123405, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.214724 -> initscore=-1.296682
[LightGBM] [Info] Start training from score -1.296682


2026/03/08 11:53:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 11:53:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data